# IELTS Writing: запуск на Kaggle

Подключите Dataset из `ielts_kaggle_bundle.zip`, включите Internet и GPU
для inference/fine_tune. Выполняйте ячейки сверху вниз. По умолчанию:
Qwen2.5-1.5B-Instruct, 20 эссе, один локальный прогон.

Параметры — во второй кодовой ячейке и корневом `RunConfig.yaml` из Dataset.
Notebook вызывает существующие классы; `main.py` и его stage_switch не запускаются.
Веса и зависимости загружаются только при выполнении на Kaggle.
Результаты находятся в `kaggle/artifacts`, кеши и временные файлы — в `kaggle/.cache`.
Notebook загружается отдельно; в Dataset нужны только код, данные и RunConfig.yaml.


## 1. Рабочая копия и временные файлы

`input_source` можно оставить пустым при одном подключённом проекте.
Поддерживаются как ZIP, так и автоматически распакованный Kaggle Dataset.


In [ ]:
import os
from pathlib import Path
import shutil
import sys
import tempfile
import zipfile

input_source = ""  # Optional: actual ZIP path or extracted project root in /kaggle/input.
project_root = Path("/kaggle/working/ielts-project")
artifacts_folder = project_root / "kaggle/artifacts"
cache_folder = project_root / "kaggle/.cache"
artifacts_folder.mkdir(parents=True, exist_ok=True)
scratch_folder = cache_folder / "scratch"
scratch_folder.mkdir(parents=True, exist_ok=True)
for variable in ("TEMP", "TMP", "TMPDIR"):
    os.environ[variable] = str(scratch_folder)
tempfile.tempdir = str(scratch_folder)
sys.dont_write_bytecode = True
cache_locations = {
    "XDG_CACHE_HOME": ".",
    "PIP_CACHE_DIR": "cache/pip",
    "HF_HOME": "cache/huggingface",
    "HF_HUB_CACHE": "cache/huggingface/hub",
    "HF_ASSETS_CACHE": "cache/huggingface/assets",
    "HF_DATASETS_CACHE": "cache/huggingface/datasets",
    "TORCH_HOME": "cache/torch",
    "TORCH_EXTENSIONS_DIR": "cache/torch-extensions",
    "TORCHINDUCTOR_CACHE_DIR": "cache/torchinductor",
    "TRITON_CACHE_DIR": "cache/triton",
    "CUDA_CACHE_PATH": "cache/cuda",
    "NUMBA_CACHE_DIR": "cache/numba",
    "MPLCONFIGDIR": "cache/matplotlib",
    "WANDB_DIR": "cache/wandb",
    "WANDB_CACHE_DIR": "cache/wandb-cache",
}
for variable, relative in cache_locations.items():
    location = cache_folder / relative.removeprefix("cache/")
    location.mkdir(parents=True, exist_ok=True)
    os.environ[variable] = str(location)
os.environ["MPLBACKEND"] = "Agg"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if input_source:
    source_path = Path(input_source)
else:
    candidates = list(Path("/kaggle/input").rglob("ielts_kaggle_bundle.zip"))
    if not candidates:
        candidates = [path.parent.parent for path in
                      Path("/kaggle/input").rglob("code/paths.yaml")]
    if len(candidates) != 1:
        raise ValueError(f"Expected one project source, found {candidates}. Set input_source.")
    source_path = candidates[0]

if source_path.is_file() and zipfile.is_zipfile(source_path):
    with zipfile.ZipFile(source_path) as archive:
        for member in archive.infolist():
            target_path = (project_root / member.filename).resolve()
            if not target_path.is_relative_to(project_root.resolve()):
                raise ValueError(f"Archive entry escapes project root: {member.filename}")
        archive.extractall(project_root)
elif source_path.is_dir():
    shutil.copytree(source_path / "code", project_root / "code", dirs_exist_ok=True)
    for filename in ("requirements.txt", "RunConfig.yaml"):
        shutil.copy2(source_path / filename, project_root / filename)
else:
    raise FileNotFoundError(f"Project source not found: {source_path}")
if not (project_root / "code/paths.yaml").is_file():
    raise FileNotFoundError("ZIP must contain code/paths.yaml at its root, without an extra src folder.")
os.chdir(project_root)
sys.path.insert(0, str(project_root / "code"))
print("Project:", project_root)
print("Experiment files:", artifacts_folder)
print("Caches:", cache_folder)


## 2. Параметры

Меняйте `mode`, модель, источник данных и ограничения ниже. Остальные настройки
находятся в корневом `RunConfig.yaml`. Повторный запуск с тем же `run_name` заменяет его результаты без суффиксов.
Для сохранения отдельного эксперимента задайте другое имя.
Дополнительные проверки выключены: `strict_checks: false`.


In [ ]:
import importlib.metadata as metadata
import json
import re
import subprocess

# Kaggle normally includes PyYAML; bootstrap it only if absent.
try:
    metadata.version("PyYAML")
except metadata.PackageNotFoundError:
    bootstrap_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "PyYAML==6.0.3"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    (cache_folder / "yaml-install.log").write_text(bootstrap_result.stdout, encoding="utf-8")
    if bootstrap_result.returncode:
        raise RuntimeError("PyYAML installation failed; see kaggle/.cache/yaml-install.log")
import yaml

configuration = yaml.safe_load((project_root / "RunConfig.yaml").read_text(encoding="utf-8"))
# Uncomment overrides here, or edit RunConfig.yaml before packaging:
# configuration["mode"] = "inference"  # inference | fine_tune | evaluate_existing
# configuration["run_name"] = "qwen25_second_run"
# configuration["model"]["id"] = "Qwen/Qwen2-1.5B-Instruct"
# configuration["model"]["secret_name"] = "HF_TOKEN"
# configuration["data"]["dataset_key"] = "custom"
# configuration["data"]["custom_csv"] = "code/resources/datasets/preprocessed_dataset.csv"
# configuration["data"]["sample_limit"] = 100
# configuration["inference"]["runs"] = 5
# configuration["inference"]["max_new_tokens"] = 384
# configuration["training"]["mode"] = "lora"
# configuration["evaluation"]["models"] = ["qwen25", "qwen2"]
# configuration["strict_checks"] = True

mode = configuration["mode"]
if mode not in {"inference", "fine_tune", "evaluate_existing"}:
    raise ValueError(f"Unknown mode: {mode}")
run_name = configuration["run_name"]
if not isinstance(run_name, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", run_name):
    raise ValueError("run_name must contain only Latin letters, digits, underscores or hyphens.")
run_folder = artifacts_folder / "runs" / run_name
strict_checks = configuration.get("strict_checks", False)
print("Experiment folder:", run_folder)
print(yaml.safe_dump(configuration, allow_unicode=True, sort_keys=False))


## 3. Зависимости выбранного режима

Установленные библиотеки используются без обновления. Из `requirements.txt`
берутся версии только отсутствующих пакетов. Ограничения pip сохраняют версии
всех уже установленных пакетов, включая NumPy и CUDA-сборку PyTorch.
Если зависимости не совместимы, установка завершается конкретной ошибкой;
требования перезапускать kernel из-за добавления нового пакета нет.


In [ ]:
package_names = {
    "numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "Pillow", "PyYAML"
}
if mode in {"inference", "fine_tune"}:
    package_names.update({"transformers", "huggingface-hub", "accelerate"})
if mode == "fine_tune":
    package_names.update({"datasets", "peft"})
requirement_lines = (project_root / "requirements.txt").read_text(encoding="utf-8").splitlines()
requested_packages = {line.split("==")[0].strip(): line.strip()
                      for line in requirement_lines
                      if line.strip() and not line.lstrip().startswith("#")
                      and line.split("==")[0].strip() in package_names}
if mode == "fine_tune" and configuration["training"]["mode"] == "qlora":
    requested_packages["bitsandbytes"] = "bitsandbytes"
    package_names.add("bitsandbytes")
installed_versions = {item.metadata["Name"].lower().replace("_", "-"): item.version
                      for item in metadata.distributions()}
missing_packages = [requirement for name, requirement in sorted(requested_packages.items())
                    if name.lower().replace("_", "-") not in installed_versions]
if mode != "evaluate_existing" and "torch" not in installed_versions:
    raise RuntimeError("Select a Kaggle GPU image with CUDA-enabled PyTorch installed.")

if missing_packages and not configuration["install_dependencies"]:
    raise RuntimeError(
        f"Missing packages: {missing_packages}. Enable install_dependencies to install only these."
    )
if missing_packages:
    constraint_path = cache_folder / "KaggleConstraints.txt"
    constraint_path.write_text("\n".join(
        f"{name}=={version}" for name, version in sorted(installed_versions.items())
    ) + "\n", encoding="utf-8")
    command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
               "--constraint", str(constraint_path), *missing_packages]
    install_result = subprocess.run(command, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True)
    install_path = cache_folder / "dependency-install.log"
    install_path.write_text(install_result.stdout, encoding="utf-8")
    print(install_result.stdout[-6000:])
    if install_result.returncode:
        raise RuntimeError(
            f"Missing-package installation failed; existing versions were constrained. "
            f"See {install_path} for the dependency conflict."
        )
    import importlib
    importlib.invalidate_caches()
else:
    print("All required packages are installed; pip was not run.")

package_versions = {}
for package in sorted(package_names | {"torch"}):
    try:
        package_versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        package_versions[package] = None
(cache_folder / "DependencySetup.json").write_text(
    json.dumps({"installed_missing": missing_packages, "versions": package_versions}, indent=2),
    encoding="utf-8",
)
print("Package versions:", json.dumps(package_versions, indent=2))


### Окружение после установки

Проверяется доступность GPU и записываются фактические версии пакетов.
Папка эксперимента создаётся после установки зависимостей и проверки GPU.
Повторное выполнение этой ячейки очищает результаты только текущего `run_name`.


In [ ]:
environment = {"python": sys.version, "packages": {}}
for package in sorted(package_names | {"torch", "bitsandbytes"}):
    try:
        environment["packages"][package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        environment["packages"][package] = None
if mode in {"inference", "fine_tune"}:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU is available. Enable GPU in Kaggle notebook settings.")
    environment["cuda"] = torch.version.cuda
    environment["gpus"] = [torch.cuda.get_device_name(index)
                           for index in range(torch.cuda.device_count())]
# A full rerun replaces this experiment; repeating later cells keeps its results.
if not run_folder.resolve().is_relative_to(artifacts_folder.resolve()):
    raise ValueError("Experiment folder escapes Kaggle artifacts")
if run_folder.exists():
    shutil.rmtree(run_folder)
(artifacts_folder / f"{run_name}.zip").unlink(missing_ok=True)
run_folder.mkdir(parents=True, exist_ok=True)
(run_folder / "RunConfig.yaml").write_text(
    yaml.safe_dump(configuration, allow_unicode=True, sort_keys=False), encoding="utf-8"
)
(run_folder / "Environment.json").write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(environment, ensure_ascii=False, indent=2))


## 4. Пути и данные

Для новых предсказаний сохраняется собственный эталон и соответствие номерам
исходных строк. Отсутствующие метки не заменяются оценками из других критериев.
Эталонные оценки, включая исходные 7.25, сохраняются без округления.
Дополнительная диагностика шага 0.5 включается только через strict_checks.
Для `evaluate_existing` используется эталон, поставляемый с историческими CSV.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from ielts_judge.paths import PATHS, load_paths
PATHS.clear()
PATHS.update(load_paths(project_root))
from ielts_judge.criteria import CRITERION_IDS

criteria = list(CRITERION_IDS)
raw_folder = run_folder / "raw"
evaluation_folder = run_folder / "evaluation"
plot_folder = run_folder / "plots"
for directory in (raw_folder, evaluation_folder, plot_folder):
    directory.mkdir(parents=True, exist_ok=True)
PATHS["results"]["llm_hugging_face_responses"] = str(raw_folder)
PATHS["results"]["llm_training"] = str(run_folder / "training")
PATHS["results"]["llm_predictions"] = str(run_folder / "fine_tuned_predictions.json")
model_identifier = configuration["model"]["id"]
model_key = re.sub(r"[^A-Za-z0-9_-]+", "_", model_identifier.rstrip("/").split("/")[-1])
if not model_key:
    raise ValueError("model.id must not be empty.")
PATHS["files"]["llm_response"] = model_key + "_{evaluation}.json"
reference_path = run_folder / "y_test.csv"

prepared_mode = mode != "evaluate_existing" and configuration["data"]["dataset_key"] == "prepared"
if mode != "evaluate_existing":
    data_settings = configuration["data"]
    dataset_key = data_settings["dataset_key"]
    if dataset_key == "prepared":
        from ielts_judge.data.preparation import load_prepared, legacy_labels
        from ielts_judge.data.runtime import export_reference, submission_records
        dataset_path = Path(PATHS["datasets"]["prepared"])
        train_frame, test_frame, dataset_manifest = load_prepared(
            dataset_path, data_settings.get("dataset_source", "combined"),
            allow_partial=data_settings.get("allow_partial", False))
        essay_column = "essay"
        for frame in (train_frame, test_frame):
            adapted = legacy_labels(frame)
            for criterion in criteria:
                frame[criterion] = adapted[criterion]
            frame["source_row"] = frame["source_line"]
        dataset = pd.concat([train_frame, test_frame], ignore_index=True)
        summary = {"dataset": str(dataset_path), "source_rows": sum(item["rows_read"] for item in dataset_manifest["sources"]),
                   "dataset_manifest": train_frame.attrs["manifest_sha256"], "partial_input": dataset_manifest["partial_input"]}
    else:
        if dataset_key == "custom":
            dataset_path = Path(data_settings["custom_csv"])
            if not dataset_path.is_absolute():
                dataset_path = project_root / dataset_path
        else:
            if dataset_key not in PATHS["datasets"]:
                raise ValueError(f"Unknown dataset key: {dataset_key}")
            dataset_path = Path(PATHS["datasets"][dataset_key])
        if not dataset_path.is_file():
            raise FileNotFoundError(f"Dataset is not included or attached: {dataset_path}")
        essay_column = data_settings["essay_column"]
        dataset = pd.read_csv(dataset_path)
        missing_columns = set([essay_column, *criteria]) - set(dataset.columns)
        if missing_columns:
            raise ValueError(f"Dataset is missing columns: {sorted(missing_columns)}")
        dataset = dataset[[essay_column, *criteria]].copy()
        dataset.insert(0, "source_row", np.arange(len(dataset)))
        summary = {"dataset": str(dataset_path), "source_rows": len(dataset)}
        essay_values = dataset[essay_column].astype("string").str.strip()
        valid_text = essay_values.notna() & essay_values.ne("")
        summary["empty_text_rows"] = int((~valid_text).sum())
        dataset = dataset.loc[valid_text].copy()
        dataset[essay_column] = essay_values.loc[valid_text]
        for criterion in criteria:
            dataset[criterion] = pd.to_numeric(dataset[criterion], errors="raise")
        missing_labels = dataset[criteria].isna().any(axis=1)
        summary["missing_label_rows"] = int(missing_labels.sum())
        if missing_labels.any() and not data_settings["drop_missing_labels"]:
            (run_folder / "DataSummary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
            raise ValueError("Missing labels found. Review data or explicitly enable drop_missing_labels.")
        dataset = dataset.loc[~missing_labels].copy()
        scores = dataset[criteria].to_numpy(dtype=float)
        valid_scores = (np.isfinite(scores) & (scores >= 0) & (scores <= 9)).all(axis=1)
        summary["invalid_score_rows"] = int((~valid_scores).sum())
        if not valid_scores.all():
            invalid_path = run_folder / "InvalidDatasetLabels.csv"
            dataset.loc[~valid_scores, ["source_row", *criteria]].to_csv(invalid_path, index=False)
            (run_folder / "DataSummary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
            raise ValueError(f"Labels must be finite numbers between 0 and 9. See {invalid_path}")
        # Source labels are not constrained to the model's requested output step.
        non_half_rows = (scores * 2 != np.rint(scores * 2)).any(axis=1)
        summary["non_half_band_rows"] = int(non_half_rows.sum())
        if strict_checks and non_half_rows.any():
            unusual_path = run_folder / "NonHalfBandLabels.csv"
            dataset.loc[non_half_rows, ["source_row", *criteria]].to_csv(unusual_path, index=False)
            print(f"Source rows with labels outside the half-band grid: {non_half_rows.sum()}. "
                  f"Original values are retained; see {unusual_path}")
        train_frame, test_frame = train_test_split(
            dataset, test_size=data_settings["test_size"], random_state=data_settings["seed"]
        )
    test_frame = test_frame.iloc[:data_settings["sample_limit"]].reset_index(drop=True)
    train_frame = train_frame.iloc[:data_settings["train_limit"]].reset_index(drop=True)
    if test_frame.empty or train_frame.empty:
        raise ValueError("Both train and test selections must be nonempty.")
    reference = test_frame[criteria].copy()
    reference.insert(0, "index", test_frame["essay_id"].to_numpy() if prepared_mode else np.arange(len(test_frame)))
    if prepared_mode:
        reference["dataset_manifest"] = test_frame.attrs["manifest_sha256"]
    reference.to_csv(reference_path, index=False)
    samples = test_frame.copy()
    samples.insert(0, "sample_id", test_frame["essay_id"].to_numpy() if prepared_mode else np.arange(len(samples)))
    samples.to_csv(run_folder / "samples.csv", index=False)
    train_frame[["essay_id"] if prepared_mode else ["source_row"]].to_csv(run_folder / "TrainingRows.csv", index=False)
    summary.update({"eligible_rows": len(dataset), "train_rows": len(train_frame),
                    "test_rows": len(test_frame), "seed": data_settings["seed"]})
    (run_folder / "DataSummary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2))
else:
    group = configuration["evaluation"]["group"]
    if group not in {"non_fine_tuned", "fine_tuned"}:
        raise ValueError("evaluation.group must be non_fine_tuned or fine_tuned.")
    source_reference = Path(PATHS["results"]["llm_evaluation_results"]) / PATHS["files"]["llm_evaluation_truth"]
    shutil.copy2(source_reference, reference_path)
    source_results = Path(PATHS["results"][f"llm_{group}_results"])
    run_order = configuration["evaluation"]["run_order"]
    result_paths = ([source_results / f"{name}.csv" for name in run_order]
                    if run_order else sorted(source_results.glob("*.csv")))
    if not result_paths:
        raise FileNotFoundError(f"No result CSVs found in {source_results}")
    selected_models = configuration["evaluation"]["models"]
    for source_csv in result_paths:
        result_frame = pd.read_csv(source_csv)
        if selected_models:
            result_frame = result_frame.loc[result_frame["model"].isin(selected_models)]
        if result_frame.empty:
            raise ValueError(f"No selected models in {source_csv.name}")
        if strict_checks and selected_models and set(selected_models) - set(result_frame["model"]):
            raise ValueError(f"Some selected models are absent from {source_csv.name}")
        result_frame.to_csv(evaluation_folder / source_csv.name, index=False)
    (run_folder / "DataSummary.json").write_text(
        json.dumps({"group": group, "source_csvs": [str(path) for path in result_paths],
                    "selected_models": selected_models}, indent=2), encoding="utf-8"
    )
    print("Selected historical CSVs:", [path.name for path in result_paths])


## 5. Локальная модель или обучение

`inference` использует существующий zero-shot промпт. PANDA/AES-подсказки p2–p6
этим notebook не включаются. `fine_tune` сохраняет PEFT-адаптер и токенизатор;
это не полная копия базовой модели. `evaluate_existing` пропускает загрузку модели.


In [ ]:
service_token = ""
if mode in {"inference", "fine_tune"}:
    secret_name = configuration["model"]["secret_name"]
    if secret_name:
        from kaggle_secrets import UserSecretsClient
        service_token = UserSecretsClient().get_secret(secret_name)
    from transformers import set_seed
    set_seed(configuration["data"]["seed"])
    test_essays = test_frame if prepared_mode else test_frame[essay_column].astype(str)

if mode == "inference":
    from ielts_judge.llm.LLMResponseObtainer import LLMResponseObtainer
    repeat_count = configuration["inference"]["runs"]
    if type(repeat_count) is not int or repeat_count < 1:
        raise ValueError("inference.runs must be a positive integer.")
    response_obtainer = LLMResponseObtainer(
        SERVICE_TOKEN=service_token,
        max_new_tokens=configuration["inference"].get("max_new_tokens", 384),
    )
    response_obtainer.load_model_and_tokenizer(model_url=model_identifier)
    response_obtainer.get_response_locally(
        number_of_evaluations=repeat_count, test_dataset=test_essays
    )

elif mode == "fine_tune":
    if configuration["training"]["mode"] not in {"lora", "qlora"}:
        raise ValueError("training.mode must be lora or qlora.")
    if service_token:
        from huggingface_hub import login
        login(service_token)
    from ielts_judge.llm.LLMFineTuner import LLMFineTuner
    fine_tuner = LLMFineTuner(
        model_name=model_identifier, mode=configuration["training"]["mode"],
        max_len=configuration["training"]["max_length"],
    )
    train_essays = submission_records(train_frame) if prepared_mode else train_frame[essay_column].astype(str).tolist()
    train_scores = train_frame[criteria].to_numpy().tolist()
    if strict_checks:
        truncated_rows = []
        for position, (essay, essay_scores) in enumerate(zip(train_essays, train_scores)):
            prompt = fine_tuner._build_prompt(essay, essay_scores)
            prompt_part, answer_part = prompt.split("### Answer:", 1)
            token_count = (len(fine_tuner.tokenizer(prompt_part + "### Answer:")["input_ids"])
                           + len(fine_tuner.tokenizer(answer_part)["input_ids"]))
            if token_count > fine_tuner.max_len:
                truncated_rows.append({**({"essay_id": train_frame.iloc[position]["essay_id"]} if prepared_mode else {"source_row": int(train_frame.iloc[position]["source_row"])}),
                                       "tokens": token_count})
        if truncated_rows:
            pd.DataFrame(truncated_rows).to_csv(run_folder / "TruncatedTrainingRows.csv", index=False)
            raise ValueError("Training would truncate target answers. Increase max_length or prepare shorter examples; see TruncatedTrainingRows.csv.")
    fine_tuner.train(train_essays, train_scores)
    adapter_folder = run_folder / "adapter"
    fine_tuner.model.save_pretrained(adapter_folder)
    fine_tuner.tokenizer.save_pretrained(adapter_folder)
    predictions = fine_tuner.evaluate_llm(
        (submission_records(test_frame) if prepared_mode else test_essays.tolist()), test_frame[criteria].to_numpy().tolist(),
        output_file=str(run_folder / "fine_tuned_predictions.json"),
    )
    (raw_folder / f"{model_key}_1.json").write_text(
        json.dumps([({"essay_id": item["essay_id"], "sample_id": item["essay_id"], "dataset_manifest": item["dataset_manifest"], "answer": item["prediction_json"]} if prepared_mode else item["prediction_json"]) for item in predictions], ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


## 6. CSV без подстановки отсутствующих ответов

Некорректные ответы сохраняются как отсутствующие значения и фиксируются
отдельно. Мода и ноль вместо ответа не подставляются. Графики по неполным
ответам будут пропущены; raw JSON и таблицы можно скачать и проверить.


In [ ]:
if mode != "evaluate_existing":
    rows = []
    response_count = configuration["inference"]["runs"] if mode == "inference" else 1
    response_paths = [raw_folder / f"{model_key}_{number}.json"
                      for number in range(1, response_count + 1)]
    for run_number, response_path in enumerate(response_paths):
        responses = json.loads(response_path.read_text(encoding="utf-8"))
        if not isinstance(responses, list) or len(responses) != len(test_frame):
            raise ValueError(f"Response count does not match the selected samples: {response_path}")
        if prepared_mode:
            ids = [item.get("essay_id") if isinstance(item, dict) else None for item in responses]
            if len(set(ids)) != len(ids) or set(ids) != set(test_frame["essay_id"]):
                raise ValueError("Prediction essay IDs do not match selected dataset samples")
        for sample_number, response in enumerate(responses):
            if prepared_mode:
                from ielts_judge.data.runtime import legacy_prediction
                scores = legacy_prediction(response.get("answer"))
            else:
                scores = response if isinstance(response, dict) else {}
            row = {"model": model_key, "file_id": run_number, "sample_id": response["essay_id"] if prepared_mode else sample_number}
            if prepared_mode:
                row["dataset_manifest"] = response["dataset_manifest"]
            for criterion in criteria:
                score = scores.get(criterion)
                row[criterion] = np.nan if isinstance(score, bool) else score
            rows.append(row)
    result_frame = pd.DataFrame(rows)
    for criterion in criteria:
        result_frame[criterion] = pd.to_numeric(result_frame[criterion], errors="coerce")
    values = result_frame[criteria].to_numpy(dtype=float)
    valid_rows = (np.isfinite(values) & (values >= 0) & (values <= 9)).all(axis=1)
    if strict_checks:
        valid_rows &= (values * 2 == np.rint(values * 2)).all(axis=1)
    invalid_frame = result_frame.loc[~valid_rows]
    invalid_frame.to_csv(run_folder / "InvalidPredictions.csv", index=False)
    experiment_key = "fine_tuned_raw_result" if mode == "fine_tune" else "non_fine_tuned_raw_result"
    result_frame.to_csv(evaluation_folder / f"{experiment_key}.csv", index=False)
    print(f"Prediction rows: {len(result_frame)}; invalid: {len(invalid_frame)}")
    reports_ready = bool(valid_rows.all())
else:
    reports_ready = True


## 7. Метрики и изображения

Используется текущий LLMEvaluator с его проверкой соответствия ID.
Дополнительная предварительная проверка выполняется только при strict_checks.
Исторический fine-tuned набор содержит неполные запуски Llama: текущий
LLMEvaluator их отклоняет, чтобы не считать отчёт по незаметно сокращённому эталону.


In [ ]:
if configuration["evaluation"]["make_plots"] and reports_ready:
    from ielts_judge.evaluation.LLMEvaluator import LLMEvaluator
    aes_metrics = None
    if mode == "evaluate_existing" and configuration["evaluation"]["aes_reference"]:
        from ielts_judge.aes.RegressorEvaluator import RegressorEvaluator
        aes_frame = pd.read_csv(PATHS["external"]["aes_predictions"])
        aes_metrics = RegressorEvaluator().get_metrics_per_criterion(
            aes_frame[[f"true_{criterion}" for criterion in criteria]].to_numpy(),
            aes_frame[[f"pred_{criterion}" for criterion in criteria]].to_numpy(), criteria,
        )
    evaluator = LLMEvaluator(
        results_folder_path=str(evaluation_folder), y_true_path=str(reference_path),
        dpi=configuration["evaluation"]["dpi"], aes_expert_metrics=aes_metrics,
    )
    report_order = (configuration["evaluation"]["run_order"] or None
                    if mode == "evaluate_existing" else [experiment_key])
    evaluator.load_all(run_order=report_order)
    if not evaluator.all_data or not evaluator._get_all_models():
        raise ValueError("No prediction runs are available for evaluation.")
    for experiment, models in evaluator.all_data.items():
        evaluator.true_names.setdefault(experiment, experiment)
        for model, runs in models.items():
            evaluator.true_model_names.setdefault(
                model, model_identifier if mode != "evaluate_existing" else model
            )
            if strict_checks:
                for run_number, frame in runs.items():
                    for criterion in criteria:
                        true_values, predicted_values = evaluator._get_true_pred(frame, criterion)
                        combined = np.concatenate([true_values, predicted_values])
                        if not (np.isfinite(combined) & (combined >= 0) & (combined <= 9)).all():
                            raise ValueError(f"Invalid scores: {experiment}, {model}, run={run_number}")
                        if not (predicted_values * 2 == np.rint(predicted_values * 2)).all():
                            raise ValueError("Predictions must follow the requested half-band output format.")
    evaluator.plot_metrics(save_dir=str(plot_folder / 'report'))
    print("Reports:", plot_folder)
elif not reports_ready:
    print("Reports skipped: invalid predictions remain. See raw/ and InvalidPredictions.csv.")
else:
    print("Plot generation is disabled in the configuration.")


## 8. Архив результата

Содержит только папку эксперимента: параметры, выборку, raw-ответы, метрики,
изображения и, для fine_tune, адаптер/checkpoints. Кеш базовой модели не входит.
После ошибки предыдущей ячейки эту ячейку можно выполнить отдельно, чтобы
скачать уже сохранённые материалы. Для постоянного хранения сохраните версию
notebook с output-файлами; одна текущая сессия не является резервной копией.


In [ ]:
result_archive = artifacts_folder / f"{run_name}.zip"
with zipfile.ZipFile(result_archive, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for result_path in sorted(run_folder.rglob("*")):
        if result_path.is_file():
            archive.write(result_path, result_path.relative_to(run_folder.parent))
print("Download from Kaggle Output:", result_archive)
print("Archive size, MB:", round(result_archive.stat().st_size / 1024**2, 2))
